In [1]:
import pandas as pd
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax

In [2]:
from datasets import Dataset
from transformers import BertTokenizer, TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import classification_report, accuracy_score

In [5]:
import evaluate
from transformers import EarlyStoppingCallback

### Setup

In [6]:
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")

In [4]:
print(df.columns)
print(df.shape)

Index(['verse_id', 'song_id', 'ori_track_name', 'clean_track_name',
       'all_artists', 'primary_artist', 'artist_genres', 'main_genre',
       'explicit', 'section', 'verse', 'language', 'language.1', 'confidence',
       'confidence.1', 'label'],
      dtype='str')
(22878, 16)


In [7]:
# Class balance check
print(df['label'].value_counts())

label
0    11439
1    11439
Name: count, dtype: int64


In [8]:
# Select only the columns we need
df = df[['verse', 'label']]

In [9]:
# Convert to Hugging Face format
dataset = Dataset.from_pandas(df)

In [10]:
# Fixes the [WinError 3] by explicitly setting a local cache directory
cache_dir = "C:/Users/User/Documents/devanasokan_fyp/huggingface_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

In [11]:
# Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir

# This turns off the annoying symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

### Tokenization

In [12]:
# Initiate tokenizer with the cache path
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', cache_dir=cache_dir)

In [13]:
def preprocess_function(examples):
    # This creates the 'input_ids' and 'attention_mask' BERT needs
    return tokenizer(examples["verse"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 22878/22878 [00:06<00:00, 3776.13 examples/s]


### Split Data (Train, Validation, Test)

In [14]:
# Ratio Used --> 70% training, 15% validation, 15% testing

# Set validation and test ratios (customizable)
desired_validation_ratio = 0.15
test_ratio = 0.15
training_ratio = 1 - desired_validation_ratio - test_ratio

# Calculate validation ratio based on the desired split
validation_ratio = desired_validation_ratio / (1 - test_ratio)  # This ensures the remaining data is split correctly

print(f"Test Ratio: {test_ratio}, Validation Ratio: {validation_ratio}")

Test Ratio: 0.15, Validation Ratio: 0.17647058823529413


In [15]:
# Split the dataset into training and testing sets
full_dataset = tokenized_dataset.train_test_split(test_size=test_ratio, seed=42)

# Split the training set into training and validation sets
train_valid_dataset = full_dataset['train'].train_test_split(test_size=validation_ratio, seed=42)

In [16]:
print("Training set size:", len(train_valid_dataset['train']))
print("Validation set size:", len(train_valid_dataset['test']))
print("Test set size:", len(full_dataset['test']))

print(train_valid_dataset) 
print(full_dataset["test"])

Training set size: 16014
Validation set size: 3432
Test set size: 3432
DatasetDict({
    train: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16014
    })
    test: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3432
    })
})
Dataset({
    features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3432
})


In [18]:
# Distribution of labels in the training set
train_labels = train_valid_dataset['train']['label']

print("Training set label distribution:")
print(pd.Series(train_labels).value_counts(normalize=True))

Training set label distribution:
1    0.500437
0    0.499563
Name: proportion, dtype: float64


### Model Training

In [19]:
model_name = "distilbert-base-uncased" # Or any model from the Hugging Face Hub

# 1. Load the tokenizer (must match the model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load the model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2293.02it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Function to initialize the model for the Trainer, useful for hyperparameter tuning
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"   # For binary classification
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
# Set training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Folder where checkpoints are saved
    eval_strategy="epoch",           # Run evaluation after every epoch
    save_strategy="epoch",           # Save model after every epoch
    learning_rate=2e-5,               # Default value; overridden in the final run
    per_device_train_batch_size=8,   # Default value; overridden in the final run
    per_device_eval_batch_size=16,    # Batch size for evaluation
    num_train_epochs=4,               # Number of epochs
    weight_decay=0.0,                # Default value; overridden in the final run
    load_best_model_at_end=True,      # Keeps the best version of the model
    metric_for_best_model="accuracy",      # Use accuracy to determine the best model
    greater_is_better=True,          # Accuracy increases with better performance
    report_to="none",
    fp16=torch.cuda.is_available()    # Use Mixed Precision if on GPU for 2x speed
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [32]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
    callbacks=[ EarlyStoppingCallback(early_stopping_patience=2) ]  # Add callback for early stopping
)

print(trainer.compute_metrics)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5426.99it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<function compute_metrics at 0x000001CB880B87C0>


### Hyperparameter Tuning

In [ ]:
# Hyperparameter search space definition
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_categorical("learning_rate", [2e-5, 3e-5, 5e-5]),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.0, 0.01, 0.1]),
    }

In [ ]:
# Run the hyperparameter search
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=20,  # Number of hyperparameter combinations to try
    compute_objective=lambda metrics: metrics["accuracy"]  # Use accuracy as the objective metric,
)

print("============================ RESULTS ============================")
print("\nBest run: ", best_run)
print("Best hyperparameters found: ", best_run.hyperparameters)

[I 2026-07-24 19:57:47,827] A new study created in memory with name: no-name-53131560-aba8-4bb6-a435-074fe4a33959
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5216.60it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.366746,0.308577,0.859848,0.850028,0.877021,0.863313
2,0.249058,0.331649,0.864510,0.908446,0.813510,0.858361
3,0.184233,0.387053,0.872960,0.895122,0.847575,0.870700
4,0.125324,0.446333,0.875291,0.880841,0.870670,0.875726


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.75it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-07-24 20:23:01,179] Trial 0 finished with value: 0.875725900116144 and parameters: {'learning_rate': 2e-05, 'per_device_train_batch_size': 32, 'weight_decay': 0.01, 'warmup_ratio': 0.05}. Best is trial 0 with value: 0.875725900116144.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5617.95it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bia

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.360393,0.300363,0.867424,0.860531,0.879908,0.870111
2,0.213687,0.332757,0.876166,0.906658,0.841224,0.872716
3,0.129081,0.483554,0.878497,0.874644,0.886259,0.880413
4,0.068633,0.589565,0.877622,0.880070,0.877021,0.878543


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-07-24 20:45:53,123] Trial 1 finished with value: 0.8785425101214575 and parameters: {'learning_rate': 5e-05, 'per_device_train_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.05}. Best is trial 1 with value: 0.8785425101214575.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3191.07it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bi

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.366866,0.308498,0.859848,0.850028,0.877021,0.863313


[I 2026-07-25 00:52:51,716] Trial 18 pruned. 
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9955.39it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.330740,0.306866,0.867716,0.870220,0.867206,0.868710


[I 2026-07-25 01:00:06,157] Trial 19 pruned. 


============================ RESULTS ============================

Best run:  BestRun(run_id='1', objective=0.8785425101214575, hyperparameters={'learning_rate': 5e-05, 'per_device_train_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.05}, run_summary=None)
Best hyperparameters found:  {'learning_rate': 5e-05, 'per_device_train_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.05}


In [ ]:
best_hyperparameters = best_run.hyperparameters
final_training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_hyperparameters["learning_rate"],
    per_device_train_batch_size=best_hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=best_hyperparameters["weight_decay"],
    warmup_ratio=best_hyperparameters["warmup_ratio"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

final_trainer = Trainer(
    model=model_init(),
    args=final_training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
    callbacks=[ EarlyStoppingCallback(early_stopping_patience=2) ]  # Add callback for early stopping
)

final_trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1456.41it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.331573,0.302088,0.867788
2,0.212709,0.326733,0.879371
3,0.162298,0.396977,0.882649


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3432, training_loss=0.24596193175771575, metrics={'train_runtime': 2218.3985, 'train_samples_per_second': 24.75, 'train_steps_per_second': 1.547, 'total_flos': 7273254990606336.0, 'train_loss': 0.24596193175771575, 'epoch': 3.0})

In [22]:
history = pd.DataFrame(final_trainer.state.log_history)
print(history)


       loss  grad_norm  learning_rate     epoch  step  eval_loss  \
0  0.390947   8.511459       0.000013  0.437063   500        NaN   
1  0.331573   7.217982       0.000011  0.874126  1000        NaN   
2       NaN        NaN            NaN  1.000000  1144   0.302088   
3  0.261531   8.955596       0.000009  1.311189  1500        NaN   
4  0.212709  10.035900       0.000006  1.748252  2000        NaN   
5       NaN        NaN            NaN  2.000000  2288   0.326733   
6  0.192129   4.305753       0.000004  2.185315  2500        NaN   
7  0.162298  17.577181       0.000002  2.622378  3000        NaN   
8       NaN        NaN            NaN  3.000000  3432   0.396977   
9       NaN        NaN            NaN  3.000000  3432        NaN   

   eval_accuracy  eval_runtime  eval_samples_per_second  \
0            NaN           NaN                      NaN   
1            NaN           NaN                      NaN   
2       0.867788       41.6599                  109.842   
3            Na

### Model Evaluation

In [ ]:
# Evaluate the model on the test set
predictions = final_trainer.predict(full_dataset["test"])

In [ ]:
# Save the predictions to a CSV file
predictions_df = pd.DataFrame({
    'verse': full_dataset["test"]["verse"],
    'true_label': predictions.label_ids,
    'predicted_label': np.argmax(predictions.predictions, axis=-1)
})
predictions_df.to_csv("C:/Users/User/Documents/devanasokan_fyp/evaluation/tuned_distilbert_predictions.csv", index=False)

In [ ]:
# --something
logits = predictions.predictions
y_true = predictions.label_ids

# Convert logits to predicted classes
y_pred = np.argmax(predictions.predictions, axis=-1)

In [ ]:
# Calculate probabilities using softmax
probabilities = softmax(logits, axis=1)

# Get the probabilities for the positive class (label 1)
y_scores = probabilities[:, 1]

In [ ]:
report = classification_report(
    y_true,
    y_pred,
    output_dict=True
)

accuracy = accuracy_score(y_true, y_pred)
precision = report["1"]["precision"]
recall = report["1"]["recall"]
f1 = report["1"]["f1-score"]

print(f"DISTILBERT CLASSIFICATION REPORT:")
print(f"Train Ratio: {training_ratio:.2f}, Validation Ratio: {desired_validation_ratio:.2f}, Test Ratio: {test_ratio:.2f}\n")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_scores)

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {roc_auc:.3f}",
    linewidth=2
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")

plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_true,
    y_scores
)

pr_auc = auc(recall, precision)

print("PR-AUC:", pr_auc)

plt.figure(figsize=(6,6))

plt.plot(
    recall,
    precision,
    label=f"PR-AUC = {pr_auc:.3f}",
    linewidth=2
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.legend(loc="lower left")
plt.grid()

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_true, y_pred)
 
# Create the plot
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True,  # Show numbers in cells
    fmt='d',     # Format as integers
    cmap='Blues',  # Color palette
    cbar=True,   # Show color bar
    xticklabels=['SAFE', 'UNSAFE'],
    yticklabels=['SAFE', 'UNSAFE']
)
 
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

### Save Model and Test

In [23]:
# Save the version currently in the final trainer's brain
final_trainer.save_model("./my_final_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]


In [24]:
# TEST MODEL

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load the model and tokenizer from your local folder
path = "./my_final_model"
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Ensure you saved the tokenizer there too!

# Create a 'pipeline' (the easiest way to use the model)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7364.39it/s]


In [25]:
# Test it on a new sentence
result = classifier("kiss it from my lips")
print(result)

[{'label': 'LABEL_1', 'score': 0.9721267223358154}]


In [34]:
import accelerate
print(accelerate.__version__)

1.13.0


In [35]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.13.0
Transformers version: 5.3.0


In [36]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
